# Basic Data Engineering
### A One-Day Intensive: 2–3 Hour Lesson + 4 Hours Hands-On
**Powered by Google Colaboratory**

Build working pipelines end-to-end: files & formats → SQL → **ETL** → API ingestion → data quality → orchestration concepts. **Core libraries:** `pandas`, `pyarrow` (Parquet), `sqlite3`, `requests`, `re`, `json`, `logging`.

---
**How to use this notebook**
1. Run cells top to bottom (`Shift + Enter` runs a cell).
2. The **Setup** section generates all dummy datasets (CSV, TXT, Parquet, and a mock API JSON) — no uploads needed.
3. Lesson sections (Parts 1 onward) are the **2–3 hour instructor-led portion**.
4. The **Hands-On Lab** section at the end is the **4-hour guided exercise portion** with a capstone.

---
> © **Data Engineering Pilipinas (DEP). All rights reserved.**
> This course material was created by and is the property of Data Engineering Pilipinas. It may not be copied, distributed, modified, or used — in whole or in part — without prior written consent from Data Engineering Pilipinas.

## ⚙️ Setup — Generate the Dummy Datasets
Run the cell below **once** at the start of the session. It creates four practice datasets in a `data/` folder:

| File | Format | Simulates |
|---|---|---|
| `sales_data.csv` | CSV | Retail sales orders (intentionally *messy*: missing values, duplicates, mixed date formats, bad emails) |
| `server_logs.txt` | TXT | Raw web-server logs (for **regex** practice) |
| `transactions.parquet` | Parquet | Bank transactions (columnar format) |
| `customers_api.json` | JSON | A saved **API response** with customer records |

In [9]:
# =============================================================
# SETUP — Run this cell first! It generates all dummy datasets.
# Creates: sales_data.csv, server_logs.txt, transactions.parquet,
#          customers_api.json  (a saved mock API response)
# =============================================================
import pandas as pd
import numpy as np
import json, random, os

np.random.seed(42)
random.seed(42)
os.makedirs("data", exist_ok=True)

# ---------- 1. CSV: retail sales data (intentionally messy) ----------
n = 500
regions  = ["NCR", "Cebu", "Davao", "Iloilo", "Baguio"]
products = ["Rice 25kg", "Cooking Oil 1L", "Instant Coffee",
            "Laundry Soap", "Canned Sardines", "Sugar 1kg"]
dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")

df = pd.DataFrame({
    "order_id":   [f"ORD-{i:05d}" for i in range(1, n + 1)],
    "order_date": np.random.choice(dates, n),
    "region":     np.random.choice(regions, n),
    "product":    np.random.choice(products, n),
    "quantity":   np.random.randint(1, 20, n),
    "unit_price": np.round(np.random.uniform(20, 1500, n), 2),
    "customer_email": [f"customer{i}@{random.choice(['gmail.com','yahoo.com','outlook.ph'])}" for i in range(n)],
    "phone":      [f"09{random.randint(100000000, 999999999)}" for _ in range(n)],
})
df["total_amount"] = (df["quantity"] * df["unit_price"]).round(2)

# Inject "messiness" so we can practice cleaning
dirty = df.copy()
dirty.loc[dirty.sample(25, random_state=1).index, "unit_price"] = np.nan
dirty.loc[dirty.sample(15, random_state=2).index, "region"] = " ncr "
dirty.loc[dirty.sample(10, random_state=3).index, "customer_email"] = "invalid-email"
dirty = pd.concat([dirty, dirty.sample(12, random_state=4)])   # duplicates
dirty["order_date"] = dirty["order_date"].astype(str)
mixed_idx = dirty.sample(20, random_state=5).index
dirty.loc[mixed_idx, "order_date"] = dirty.loc[mixed_idx, "order_date"].str.replace("-", "/")
dirty.to_csv("data/sales_data.csv", index=False)

# ---------- 2. TXT: server log file (for regex practice) ----------
levels    = ["INFO", "WARNING", "ERROR", "DEBUG"]
endpoints = ["/api/orders", "/api/customers", "/api/products", "/login", "/checkout"]
lines = []
for i in range(300):
    ts  = pd.Timestamp("2025-06-01") + pd.Timedelta(seconds=int(np.random.uniform(0, 86400 * 30)))
    lvl = random.choices(levels, weights=[60, 15, 10, 15])[0]
    ip  = f"{random.randint(1,255)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
    ep  = random.choice(endpoints)
    ms  = random.randint(5, 2500)
    status = random.choices([200, 201, 404, 500, 403], weights=[70, 10, 8, 7, 5])[0]
    lines.append(f"{ts.strftime('%Y-%m-%d %H:%M:%S')} [{lvl}] client={ip} "
                 f"endpoint={ep} status={status} response_time={ms}ms")
with open("data/server_logs.txt", "w") as f:
    f.write("\n".join(sorted(lines)))

# ---------- 3. Parquet: bank-style transactions ----------
m = 800
tx = pd.DataFrame({
    "transaction_id":   [f"TXN{i:07d}" for i in range(1, m + 1)],
    "account_id":       [f"ACC{random.randint(1000,1099)}" for _ in range(m)],
    "transaction_type": np.random.choice(["deposit","withdrawal","transfer","payment"], m, p=[.3,.3,.2,.2]),
    "amount":           np.round(np.random.lognormal(mean=7, sigma=1.2, size=m), 2),
    "channel":          np.random.choice(["mobile_app","atm","branch","online"], m),
    "timestamp":        pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.uniform(0,365,m), unit="D"),
    "is_flagged":       np.random.choice([True, False], m, p=[.04,.96]),
})
tx.to_parquet("data/transactions.parquet", index=False)

# ---------- 4. Mock API response saved as JSON ----------
first = ["Maria","Jose","Juan","Ana","Carlo","Liza","Ramon","Grace","Paolo","Nina"]
last  = ["Santos","Reyes","Cruz","Garcia","Torres","Flores","Ramos","Mendoza"]
customers = []
for i in range(1, 61):
    fn, ln = random.choice(first), random.choice(last)
    customers.append({
        "customer_id": i,
        "name": f"{fn} {ln}",
        "email": f"{fn.lower()}.{ln.lower()}{i}@example.com",
        "city": random.choice(["Quezon City","Makati","Cebu City","Davao City","Taguig"]),
        "signup_date": str((pd.Timestamp("2024-01-01") + pd.Timedelta(days=random.randint(0,700))).date()),
        "loyalty_points": random.randint(0, 5000),
        "is_active": random.random() > 0.15,
    })
with open("data/customers_api.json", "w") as f:
    json.dump({"status": "success", "count": len(customers), "data": customers}, f, indent=2)

print("✅ Setup complete! Files created in ./data/:")
for fn in sorted(os.listdir("data")):
    print("   -", fn)

✅ Setup complete! Files created in ./data/:
   - .ipynb_checkpoints
   - customers_api.json
   - sales_clean.csv
   - sales_clean.parquet
   - sales_data.csv
   - server_logs.txt
   - transactions.parquet


## Part 1 — Python Essentials (≈30 min)
Python is the *lingua franca* of data work. We only need a focused subset: **variables, collections, control flow, functions**, and the habit of writing small, reusable pieces of logic.

In [10]:
# Variables & data types
store_name = "DEP Sari-Sari Store"   # str
daily_sales = 15750.50               # float
num_customers = 42                   # int
is_open = True                       # bool

print(type(store_name), type(daily_sales), type(num_customers), type(is_open))
print(f"{store_name} earned ₱{daily_sales:,.2f} from {num_customers} customers today.")

<class 'str'> <class 'float'> <class 'int'> <class 'bool'>
DEP Sari-Sari Store earned ₱15,750.50 from 42 customers today.


In [11]:
# Collections: list, tuple, dict, set
products = ["rice", "oil", "coffee", "soap"]            # list  — ordered, mutable
location = (14.6760, 121.0437)                          # tuple — ordered, immutable (e.g., lat/lon)
prices   = {"rice": 52.0, "oil": 85.5, "coffee": 7.5}   # dict  — key/value lookup
regions  = {"NCR", "Cebu", "NCR", "Davao"}              # set   — unique values only

print("3rd product:", products[2])
print("Price of oil:", prices["oil"])
print("Unique regions:", regions)

3rd product: coffee
Price of oil: 85.5
Unique regions: {'NCR', 'Davao', 'Cebu'}


In [12]:
# Control flow: if / for / while
sales = [1200, 3400, 800, 5600, 2300]

total = 0
for amount in sales:
    total += amount
    if amount > 3000:
        print(f"  High-value sale: ₱{amount:,}")

print(f"Total: ₱{total:,}")
print(f"Average: ₱{total/len(sales):,.2f}")

# List comprehension — the "pythonic" one-liner loop
vat_inclusive = [round(s * 1.12, 2) for s in sales]
print("With 12% VAT:", vat_inclusive)

  High-value sale: ₱3,400
  High-value sale: ₱5,600
Total: ₱13,300
Average: ₱2,660.00
With 12% VAT: [1344.0, 3808.0, 896.0, 6272.0, 2576.0]


In [13]:
# Functions + error handling — the building blocks of reusable data code
def compute_discount(amount, customer_type="regular"):
    """Return the discounted amount based on customer type."""
    rates = {"regular": 0.0, "senior": 0.20, "pwd": 0.20, "member": 0.05}
    if customer_type not in rates:
        raise ValueError(f"Unknown customer type: {customer_type}")
    return round(amount * (1 - rates[customer_type]), 2)

print(compute_discount(1000))             # 1000.0
print(compute_discount(1000, "senior"))   # 800.0

# try/except keeps pipelines and analyses from crashing on bad input
try:
    compute_discount(1000, "vip")
except ValueError as e:
    print("Handled gracefully →", e)

1000.0
800.0
Handled gracefully → Unknown customer type: vip


In [14]:
# Engineers work with raw files constantly — reading & writing without pandas
# TXT
with open("data/server_logs.txt") as f:
    first_lines = [next(f) for _ in range(3)]
print("".join(first_lines))

# JSON
import json
with open("data/customers_api.json") as f:
    payload = json.load(f)
print("API status:", payload["status"], "| records:", payload["count"])
print("First record:", payload["data"][0])

2025-06-01 08:55:51 [ERROR] client=40.199.162.93 endpoint=/api/orders status=200 response_time=379ms
2025-06-01 09:04:38 [INFO] client=137.47.86.152 endpoint=/checkout status=200 response_time=636ms
2025-06-01 12:57:52 [INFO] client=204.203.40.96 endpoint=/login status=200 response_time=1885ms

API status: success | records: 60
First record: {'customer_id': 1, 'name': 'Carlo Mendoza', 'email': 'carlo.mendoza1@example.com', 'city': 'Quezon City', 'signup_date': '2024-08-21', 'loyalty_points': 1100, 'is_active': True}


## Regular Expressions (Regex) — Pattern Matching for Text (≈25 min)
Regex lets you **find, validate, and extract** patterns in text — emails, phone numbers, log entries, IDs. The `re` module is built into Python, and pandas has `.str.extract()` / `.str.contains()` built on top of it.

**Cheat sheet:**
| Pattern | Matches |
|---|---|
| `\d` | a digit (0–9) |
| `\w` | a word character (letter, digit, `_`) |
| `\s` | whitespace |
| `+` / `*` | one-or-more / zero-or-more |
| `{n,m}` | between *n* and *m* repeats |
| `[abc]` | any one of a, b, c |
| `^` / `$` | start / end of string |
| `( )` | capture group (extract this part) |

In [15]:
import re

# --- Extract structured fields from raw log lines ---
with open("data/server_logs.txt") as f:
    logs = f.readlines()

print("Raw log line:")
print(logs[0])

# One pattern, four capture groups: timestamp, level, status, response time
pattern = r"^([\d\-]+ [\d:]+) \[(\w+)\] client=([\d\.]+) endpoint=(\S+) status=(\d{3}) response_time=(\d+)ms"

m = re.match(pattern, logs[0])
print("\nParsed:", m.groups())

# Parse ALL lines into a DataFrame — regex turns unstructured text into a table!
import pandas as pd
parsed = [re.match(pattern, line).groups() for line in logs if re.match(pattern, line)]
log_df = pd.DataFrame(parsed, columns=["timestamp", "level", "client_ip", "endpoint", "status", "response_ms"])
log_df["response_ms"] = log_df["response_ms"].astype(int)
log_df["status"] = log_df["status"].astype(int)
log_df.head()

Raw log line:
2025-06-01 08:55:51 [ERROR] client=40.199.162.93 endpoint=/api/orders status=200 response_time=379ms


Parsed: ('2025-06-01 08:55:51', 'ERROR', '40.199.162.93', '/api/orders', '200', '379')


,timestamp,level,client_ip,endpoint,status,response_ms
0,2025-06-01 08:55:51,ERROR,40.199.162.93,/api/orders,200,379
1,2025-06-01 09:04:38,INFO,137.47.86.152,/checkout,200,636
2,2025-06-01 12:57:52,INFO,204.203.40.96,/login,200,1885
3,2025-06-01 15:42:46,INFO,21.168.194.245,/api/products,404,2003
4,2025-06-01 19:33:37,INFO,39.31.230.27,/api/products,200,350


In [16]:
# --- Validating & extracting with regex ---
import re

emails = ["nina@dep.org.ph", "invalid-email", "juan.cruz@gmail.com", "test@", "ana_reyes@yahoo.com"]
email_pattern = r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$"

for e in emails:
    ok = "✅ valid" if re.match(email_pattern, e) else "❌ invalid"
    print(f"{e:30s} {ok}")

# Extract PH mobile numbers from free text
text = "Contact us at 09171234567 or 09989876543. Landline: 8123-4567."
print("\nMobile numbers found:", re.findall(r"09\d{9}", text))

# re.sub — find & replace (mask sensitive digits)
masked = re.sub(r"(09\d{2})\d{5}(\d{2})", r"\1*****\2", text)
print("Masked:", masked)

nina@dep.org.ph                ✅ valid
invalid-email                  ❌ invalid
juan.cruz@gmail.com            ✅ valid
test@                          ❌ invalid
ana_reyes@yahoo.com            ✅ valid

Mobile numbers found: ['09171234567', '09989876543']
Masked: Contact us at 0917*****67 or 0998*****43. Landline: 8123-4567.


In [17]:
# --- Regex inside pandas: clean a real column ---
import pandas as pd
sales = pd.read_csv("data/sales_data.csv")

# Flag invalid emails using .str.match()
valid = sales["customer_email"].str.match(r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$")
print(f"Invalid emails found: {(~valid).sum()} out of {len(sales)} rows")
sales.loc[~valid, "customer_email"].head()

Invalid emails found: 10 out of 512 rows


27     invalid-email
37     invalid-email
130    invalid-email
191    invalid-email
211    invalid-email
Name: customer_email, dtype: str

## Part 3 — Data Formats & Schemas (≈20 min)
| Format | Type | Strengths | Weaknesses |
|---|---|---|---|
| **CSV** | Row, text | Universal, human-readable | No types, no schema, big files |
| **JSON** | Nested, text | APIs, flexible structure | Verbose, slow at scale |
| **TXT** | Raw text | Logs, anything | Needs parsing (regex!) |
| **Parquet** | Columnar, binary | Fast, compressed, typed | Not human-readable |

**Schema** = the contract: column names + types. Parquet stores it; CSV loses it.

In [18]:
import pandas as pd
import os

# Same data, three formats — compare size & type fidelity
tx = pd.read_parquet("data/transactions.parquet")
tx.to_csv("data/tx_copy.csv", index=False)
tx.to_json("data/tx_copy.json", orient="records")

for f in ["data/transactions.parquet", "data/tx_copy.csv", "data/tx_copy.json"]:
    print(f"{f:35s} {os.path.getsize(f)/1024:8.1f} KB")

# Schema fidelity: reload the CSV — the timestamp became a plain string!
tx_csv = pd.read_csv("data/tx_copy.csv")
print("\nParquet timestamp dtype:", tx["timestamp"].dtype)
print("CSV     timestamp dtype:", tx_csv["timestamp"].dtype, "← schema lost, must re-parse")

data/transactions.parquet               23.3 KB
data/tx_copy.csv                        61.5 KB
data/tx_copy.json                      129.4 KB

Parquet timestamp dtype: datetime64[ns]
CSV     timestamp dtype: str ← schema lost, must re-parse


/var/folders/qh/05_lrvnj2cgcqwdk8n7l1cwm0000gn/T/ipykernel_2834/626601529.py:7: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  tx.to_json("data/tx_copy.json", orient="records")


## Part 4 — SQL with SQLite (≈30 min)
Every data engineer speaks SQL. SQLite is a full SQL database in a single file — perfect for practice, and it runs natively in Colab.

In [24]:
import sqlite3
import pandas as pd

# Create a database and load our tables into it
conn = sqlite3.connect("dep_warehouse.db")

sales = pd.read_csv("data/sales_data.csv")
tx = pd.read_parquet("data/transactions.parquet")
import json
with open("data/customers_api.json") as f:
    customers = pd.json_normalize(json.load(f)["data"])

sales.to_sql("sales", conn, if_exists="replace", index=False)
tx.to_sql("transactions", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

           name
0   sales_clean
1   sales_daily
2         sales
3  transactions
4     customers


In [25]:
# Core SQL: SELECT, WHERE, GROUP BY, ORDER BY, JOIN
import pandas as pd

q1 = """
SELECT region,
       COUNT(*)            AS orders,
       ROUND(SUM(total_amount), 2) AS revenue,
       ROUND(AVG(total_amount), 2) AS avg_order
FROM sales
WHERE quantity >= 5
GROUP BY region
ORDER BY revenue DESC;
"""
print(pd.read_sql(q1, conn))

q2 = """
SELECT c.city,
       COUNT(t.transaction_id) AS txn_count,
       ROUND(SUM(t.amount), 2) AS total_amount
FROM transactions t
JOIN customers c
  ON (CAST(SUBSTR(t.account_id, 4) AS INT) - 999) = c.customer_id  -- toy join key
GROUP BY c.city
ORDER BY total_amount DESC;
"""
pd.read_sql(q2, conn)

   region  orders    revenue  avg_order
0  Baguio      87  887107.61   10196.64
1  Iloilo      80  758964.36    9487.05
2   Davao      72  707219.95    9822.50
3     NCR      83  688275.21    8292.47
4    Cebu      75  579161.28    7722.15
5    ncr       12  114282.44    9523.54


,city,txn_count,total_amount
0,Davao City,110,258347.30
1,Makati,109,232801.10
2,Cebu City,117,221230.75
3,Taguig,85,200924.66
4,Quezon City,78,137185.23


## Part 5 — Building an ETL Pipeline (≈40 min)
**E**xtract → **T**ransform → **L**oad, written as **functions** so each step is testable and reusable. Add `logging` so the pipeline tells you what it's doing — `print()` doesn't scale.

In [7]:
import pandas as pd
import sqlite3
import logging
import re

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("etl")

# ---------- EXTRACT ----------
def extract(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    log.info(f"Extracted {len(df)} rows from {csv_path}")
    return df

# ---------- TRANSFORM ----------
def transform(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates()
    df["region"] = df["region"].str.strip().str.upper()
    mask = df["unit_price"].isna()
    df.loc[mask, "unit_price"] = (df.loc[mask, "total_amount"] / df.loc[mask, "quantity"]).round(2)
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")
    df["email_valid"] = df["customer_email"].str.match(r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$")
    log.info(f"Transformed: {before} → {len(df)} rows "
             f"({before - len(df)} duplicates removed, {mask.sum()} prices repaired)")
    return df

# ---------- LOAD ----------
def load(df: pd.DataFrame, db_path: str, table: str) -> int:
    with sqlite3.connect(db_path) as conn:
        df.to_sql(table, conn, if_exists="replace", index=False)
    log.info(f"Loaded {len(df)} rows into {db_path}:{table}")
    return len(df)

# ---------- RUN THE PIPELINE ----------
raw = extract("data/sales_data.csv")
clean = transform(raw)
load(clean, "dep_warehouse.db", "sales_clean")
clean.to_parquet("data/sales_clean.parquet", index=False)   # also land it in the "lake"
log.info("✅ Pipeline complete")

2026-07-11 17:39:59,102 [INFO] Extracted 512 rows from data/sales_data.csv
2026-07-11 17:39:59,146 [INFO] Transformed: 512 → 500 rows (12 duplicates removed, 25 prices repaired)
2026-07-11 17:39:59,156 [INFO] Loaded 500 rows into dep_warehouse.db:sales_clean
2026-07-11 17:39:59,187 [INFO] ✅ Pipeline complete


## Part 6 — Ingesting Data from APIs (≈25 min)
The standard ingestion pattern: **request → check status → parse JSON → normalize → land it** (save to storage). Real pipelines also handle **pagination**, **rate limits**, and **retries**.

In [19]:
import requests
import pandas as pd
import json

def ingest_api(url: str, fallback_file: str) -> pd.DataFrame:
    """Fetch JSON from an API with a local fallback (offline-safe)."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        records = resp.json()
        print(f"✅ API OK ({resp.status_code}) — {len(records)} records from {url}")
    except Exception as e:
        print(f"⚠️ API unavailable ({e}); using fallback file")
        with open(fallback_file) as f:
            records = json.load(f)["data"]
    return pd.json_normalize(records)

users = ingest_api("https://jsonplaceholder.typicode.com/users", "data/customers_api.json")

# Land the raw ingest as Parquet — "raw zone" of a data lake
users.to_parquet("data/api_users_raw.parquet", index=False)
users.head(3)

✅ API OK (200) — 10 records from https://jsonplaceholder.typicode.com/users


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications


In [20]:
# Pagination pattern — most real APIs return data in pages
import requests

all_posts = []
for page in range(1, 4):                       # fetch 3 pages
    r = requests.get("https://jsonplaceholder.typicode.com/posts",
                     params={"_page": page, "_limit": 10}, timeout=10)
    if r.status_code != 200:
        print("Stopping: status", r.status_code); break
    batch = r.json()
    if not batch:
        break                                   # no more data
    all_posts.extend(batch)
    print(f"Page {page}: +{len(batch)} records (total {len(all_posts)})")

Page 1: +10 records (total 10)
Page 2: +10 records (total 20)
Page 3: +10 records (total 30)


## Part 7 — Data Quality & Validation (≈25 min)
A pipeline that silently loads bad data is worse than one that fails loudly. Classic dimensions: **completeness, uniqueness, validity, consistency, timeliness**.

In [21]:
import pandas as pd

def validate(df: pd.DataFrame) -> dict:
    """Run a data quality report; return dict of check → pass/fail."""
    checks = {
        "no_nulls_in_key":      df["order_id"].notna().all(),
        "order_id_unique":      not df["order_id"].duplicated().any(),
        "quantity_positive":    (df["quantity"] > 0).all(),
        "price_positive":       (df["unit_price"] > 0).all(),
        "total_consistent":     ((df["quantity"] * df["unit_price"]).round(2)
                                  .sub(df["total_amount"]).abs() < 0.05).all(),
        "valid_regions":        df["region"].isin(["NCR","CEBU","DAVAO","ILOILO","BAGUIO"]).all(),
        "dates_in_range":       df["order_date"].between("2025-01-01", "2025-12-31").all(),
    }
    return checks

clean = pd.read_parquet("data/sales_clean.parquet")
report = validate(clean)

print("DATA QUALITY REPORT")
print("=" * 40)
failed = 0
for check, passed in report.items():
    print(f"{'✅ PASS' if passed else '❌ FAIL':8s} {check}")
    failed += (not passed)
print("=" * 40)
if failed:
    raise ValueError(f"{failed} data quality check(s) failed — blocking load!")
print("All checks passed — safe to load downstream. 🚀")

DATA QUALITY REPORT
✅ PASS   no_nulls_in_key
✅ PASS   order_id_unique
✅ PASS   quantity_positive
✅ PASS   price_positive
✅ PASS   total_consistent
✅ PASS   valid_regions
✅ PASS   dates_in_range
All checks passed — safe to load downstream. 🚀


## Part 8 — Orchestration Concepts (≈15 min)
Production pipelines are **DAGs** (Directed Acyclic Graphs) of tasks, run on a schedule by tools like **Apache Airflow** or **AWS Step Functions**. We can simulate the core idea — ordered tasks with dependencies and failure handling — in plain Python.

In [22]:
import logging, time
log = logging.getLogger("orchestrator")

# A mini "DAG": each task depends on the previous one succeeding
def task(name, fn):
    log.info(f"▶ starting task: {name}")
    t0 = time.time()
    try:
        result = fn()
        log.info(f"✔ finished {name} in {time.time()-t0:.2f}s")
        return result
    except Exception as e:
        log.error(f"✖ task {name} FAILED: {e}")
        raise   # fail fast — downstream tasks must not run on bad data

import pandas as pd
pipeline_state = {}

task("extract",   lambda: pipeline_state.update(raw=extract("data/sales_data.csv")))
task("transform", lambda: pipeline_state.update(clean=transform(pipeline_state["raw"])))
task("validate",  lambda: validate(pipeline_state["clean"]))
task("load",      lambda: load(pipeline_state["clean"], "dep_warehouse.db", "sales_daily"))

print("""
In production this becomes an Airflow DAG:
  extract >> transform >> validate >> load
scheduled e.g. daily at 02:00, with retries, alerts, and backfills.""")

2026-07-11 21:09:17,698 [INFO] ▶ starting task: extract
2026-07-11 21:09:17,705 [INFO] Extracted 512 rows from data/sales_data.csv
2026-07-11 21:09:17,705 [INFO] ✔ finished extract in 0.01s
2026-07-11 21:09:17,706 [INFO] ▶ starting task: transform
2026-07-11 21:09:17,727 [INFO] Transformed: 512 → 500 rows (12 duplicates removed, 25 prices repaired)
2026-07-11 21:09:17,727 [INFO] ✔ finished transform in 0.02s
2026-07-11 21:09:17,728 [INFO] ▶ starting task: validate
2026-07-11 21:09:17,730 [INFO] ✔ finished validate in 0.00s
2026-07-11 21:09:17,731 [INFO] ▶ starting task: load
2026-07-11 21:09:17,743 [INFO] Loaded 500 rows into dep_warehouse.db:sales_daily
2026-07-11 21:09:17,743 [INFO] ✔ finished load in 0.01s



In production this becomes an Airflow DAG:
  extract >> transform >> validate >> load
scheduled e.g. daily at 02:00, with retries, alerts, and backfills.


---
# 🧪 HANDS-ON LAB (4 hours)

### Block A — Files, Formats & Regex (⭐ 45 min)
1. Convert `data/customers_api.json` to both CSV and Parquet; compare file sizes.
2. Parse `data/server_logs.txt` with regex into a DataFrame; save it as `data/logs.parquet`.
3. From the parsed logs: what % of requests are errors (status ≥ 400), and which `client_ip` appears most often?

In [26]:
# Block A — your code here
import pandas as pd
import json
import os

with open("data/customers_api.json") as f:
    customers = pd.json_normalize(json.load(f)["data"])

customers.to_csv("data/customers.csv",index=False)
customers.to_parquet("data/customers.parquet",index=False)

csv_size = os.path.getsize("data/customers.csv")
parquet_size = os.path.getsize("data/customers.parquet")

print("File size comparison")
print(f"CSV:     {csv_size:,} bytes ({csv_size / 1024:.2f} KB)")
print(f"Parquet: {parquet_size:,} bytes ({parquet_size / 1024:.2f} KB)")

if csv_size < parquet_size:
    print("\nResult: CSV file used less storage space than the Parquet file.")
elif parquet_size < csv_size:
    print("\nResult: Parquet file used less storage space than the CSV file.")
else:
    print("\nResult: Both files have the same size.\n")

print("\n-----------------------------------------------------")


with open("data/server_logs.txt") as f:
    logs = f.readlines()

pattern = r"^([\d\-]+ [\d:]+) \[(\w+)\] client=([\d\.]+) endpoint=(\S+) status=(\d{3}) response_time=(\d+)ms"

parsed = []

for line in logs:
    match = re.match(pattern, line)

    if match:
        parsed.append(match.groups())
        
log_df = pd.DataFrame(parsed, columns=["timestamp", "level", "client_ip", "endpoint", "status", "response_ms"])
log_df["response_ms"] = log_df["response_ms"].astype(int)
log_df["status"] = log_df["status"].astype(int)
log_df.to_parquet("data/logs.parquet", index=False)
log_df.head()


error_count = (log_df["status"] >= 400).sum()
error_percentage = (error_count / len(log_df)) * 100
IP_counts = log_df["client_ip"].value_counts()
highest_count = IP_counts.max()
most_IPs = IP_counts[IP_counts == highest_count]

print(f"Error requests: {error_count}")
print(f"Total requests: {len(log_df)}")
print(f"Error percentage: {error_percentage:.2f}%")

print("-----------------------------------------------------")

print("\nClient IP address that appears most frequently:\n")
print(most_IPs)


print("\nNo client IP appeared most frequently because all 300 client IP addresses occurred only once.")

File size comparison
CSV:     4,348 bytes (4.25 KB)
Parquet: 6,580 bytes (6.43 KB)

Result: CSV file used less storage space than the Parquet file.

-----------------------------------------------------
Error requests: 59
Total requests: 300
Error percentage: 19.67%
-----------------------------------------------------

Client IP address that appears most frequently:

client_ip
40.199.162.93     1
137.47.86.152     1
204.203.40.96     1
21.168.194.245    1
39.31.230.27      1
                 ..
180.151.11.251    1
80.203.242.65     1
203.71.121.203    1
15.188.146.20     1
129.77.176.7      1
Name: count, Length: 300, dtype: int64

No client IP appeared most frequently because all 300 client IP addresses occurred only once.


### Block B — SQL (⭐⭐ 60 min)
Using `dep_warehouse.db`:
1. Top 3 products by revenue **per region** (hint: window functions `ROW_NUMBER() OVER (PARTITION BY ...)` or a subquery).
2. Monthly transaction totals from the `transactions` table (`strftime('%Y-%m', timestamp)`).
3. Which accounts have **more than 12 transactions**? List them with their total amount.
4. Load your parsed logs table into SQLite and find the 5 slowest endpoints by average response time.

In [31]:
# Block B — your code here
import sqlite3
import pandas as pd

conn = sqlite3.connect("dep_warehouse.db")

log_df.to_sql(
    "logs",
    conn,
    if_exists="replace",
    index=False
)

top_products_query = """
WITH product_revenue AS (
    SELECT
        UPPER(TRIM(region)) AS region,
        product,
        ROUND(SUM(total_amount), 2) AS revenue
    FROM sales
    GROUP BY
        UPPER(TRIM(region)),
        product
),

ranked_products AS (
    SELECT
        region,
        product,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY region
            ORDER BY revenue DESC
        ) AS revenue_rank
    FROM product_revenue
)

SELECT
    region,
    revenue_rank AS rank,
    product,
    revenue
FROM ranked_products
WHERE revenue_rank <= 3
ORDER BY region, revenue_rank;
"""
print("\n")
print(pd.read_sql(top_products_query, conn))

monthly_totals_query = """
SELECT
    strftime('%Y-%m', timestamp) AS month,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount), 2) AS total_transaction_amount
FROM transactions
GROUP BY strftime('%Y-%m', timestamp)
ORDER BY month;
"""
print("\n")
print(pd.read_sql(monthly_totals_query, conn))

accounts_query = """
SELECT
    account_id,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount), 2) AS total_amount
FROM transactions
GROUP BY account_id
HAVING COUNT(*) > 12
ORDER BY transaction_count DESC, total_amount DESC;
"""
print("\n")
print(pd.read_sql(accounts_query, conn))

slowest_endpoints_query = """
SELECT
    endpoint,
    COUNT(*) AS request_count,
    ROUND(AVG(response_ms), 2) AS average_response_ms
FROM logs
GROUP BY endpoint
ORDER BY average_response_ms DESC
LIMIT 5;
"""
print("\n")
print(pd.read_sql(slowest_endpoints_query,conn))

conn.close()

print("\n")




    region  rank          product    revenue
0   BAGUIO     1   Cooking Oil 1L  205090.89
1   BAGUIO     2   Instant Coffee  174214.59
2   BAGUIO     3  Canned Sardines  155861.64
3     CEBU     1   Cooking Oil 1L  151400.66
4     CEBU     2  Canned Sardines  123259.54
5     CEBU     3        Sugar 1kg  108944.63
6    DAVAO     1        Rice 25kg  143982.30
7    DAVAO     2        Sugar 1kg  138192.85
8    DAVAO     3  Canned Sardines  137424.87
9   ILOILO     1   Instant Coffee  204484.65
10  ILOILO     2   Cooking Oil 1L  148913.25
11  ILOILO     3        Rice 25kg  131637.25
12     NCR     1        Sugar 1kg  172163.41
13     NCR     2     Laundry Soap  158681.40
14     NCR     3        Rice 25kg  144525.31


      month  transaction_count  total_transaction_amount
0   2025-01                 63                 124767.66
1   2025-02                 55                  93545.93
2   2025-03                 76                 149146.18
3   2025-04                 69                 1

### Block C — Build Your Own ETL (⭐⭐⭐ 75 min)
Build a pipeline `etl_transactions()` that:
1. **Extracts** `data/transactions.parquet`
2. **Transforms**: adds `amount_bucket` (small < ₱1k ≤ medium < ₱10k ≤ large), adds `txn_hour` from timestamp, filters out negative/zero amounts
3. **Validates**: unique `transaction_id`, all amounts positive, timestamps within 2025 — raise on failure
4. **Loads** into SQLite table `transactions_curated` AND writes a Parquet copy
5. Uses `logging` (not print) throughout, and wraps steps in the `task()` orchestrator pattern.

In [34]:
# Block C — your code here
import pandas as pd
import sqlite3
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("etl_transactions")


def task(name, fn):
    """Run one pipeline task with logging and error handling."""

    start_time = time.time()
    log.info(f"Starting task: {name}")

    try:
        result = fn()
        elapsed = time.time() - start_time
        log.info(f"Completed task: {name} in {elapsed:.2f} seconds")
        return result

    except Exception:
        log.exception(f"Task failed: {name}")
        raise


def extract(file_path: str = "data/transactions.parquet") -> pd.DataFrame:
    """Extract transaction data from a Parquet file."""

    log.info(f"Extracting transactions from {file_path}")
    transactions = pd.read_parquet(file_path)
    log.info(f"Successfully extracted {len(transactions)} transaction records")

    return transactions


def transform(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and transform the transaction data."""

    log.info("Starting transaction transformation")

    transformed = df.copy()
    transformed["timestamp"] = pd.to_datetime(transformed["timestamp"], errors="coerce")

    rows_before = len(transformed)

    transformed = transformed[transformed["amount"] > 0].copy()

    rows_removed = rows_before - len(transformed)

    transformed["amount_bucket"] = pd.cut(
        transformed["amount"],
        bins=[float("-inf"), 1000, 10000, float("inf")],
        labels=["small", "medium", "large"],
        right=False
    ).astype("string")

    transformed["txn_hour"] = transformed["timestamp"].dt.hour

    log.info(f"Transformation completed. Removed {rows_removed} invalid transactions.")

    return transformed


def validate(df: pd.DataFrame) -> bool:
    """Validate the cleaned transaction data."""

    log.info("Starting transaction validation")

    checks = {
        "transaction_id is unique": df["transaction_id"].is_unique,
        "all amounts are positive": (df["amount"] > 0).all(),
        "all timestamps are valid": df["timestamp"].notna().all(),
        "all timestamps are within 2025": df["timestamp"].between("2025-01-01", "2025-12-31 23:59:59").all()
    }

    failed_checks = []

    for check_name, passed in checks.items():
        if passed:
            log.info(f"PASS: {check_name}")
        else:
            log.error(f"FAIL: {check_name}")
            failed_checks.append(check_name)

    if failed_checks:
        raise ValueError("Transaction validation failed: " + ", ".join(failed_checks))

    log.info("All transaction validation checks passed")

    return True


def load_transactions_sqlite(df: pd.DataFrame, db_path: str, table_name: str) -> int:
    """Load transformed transactions into SQLite."""

    with sqlite3.connect(db_path) as conn:
        df.to_sql(table_name, conn, if_exists="replace", index=False)

    log.info(f"Loaded {len(df)} rows into {db_path}:{table_name}")

    return len(df)


def write_transactions_parquet(df: pd.DataFrame, output_path: str) -> int:
    """Save transformed transactions as a Parquet file."""

    df.to_parquet(output_path, index=False)

    log.info(f"Saved {len(df)} rows to {output_path}")

    return len(df)


def etl_transactions() -> pd.DataFrame:
    """Run the complete transaction ETL pipeline."""

    source_path = "data/transactions.parquet"
    database_path = "dep_warehouse.db"
    parquet_output = "data/transactions_curated.parquet"
    table_name = "transactions_curated"

    log.info("Transaction ETL pipeline started")

    raw_transactions = task("Extract transactions", lambda: extract(source_path))

    curated_transactions = task("Transform transactions", lambda: transform(raw_transactions))

    task("Validate transactions", lambda: validate(curated_transactions))

    task(
        "Load transactions into SQLite",
        lambda: load_transactions_sqlite(curated_transactions, database_path, table_name)
    )

    task(
        "Write Parquet copy",
        lambda: write_transactions_parquet(curated_transactions, parquet_output)
    )

    log.info("Transaction ETL pipeline completed successfully")

    return curated_transactions


curated_transactions = etl_transactions()

curated_transactions.head()

2026-07-11 23:58:17,683 [INFO] Transaction ETL pipeline started
2026-07-11 23:58:17,684 [INFO] Starting task: Extract transactions
2026-07-11 23:58:17,684 [INFO] Extracting transactions from data/transactions.parquet
2026-07-11 23:58:17,688 [INFO] Successfully extracted 800 transaction records
2026-07-11 23:58:17,689 [INFO] Completed task: Extract transactions in 0.01 seconds
2026-07-11 23:58:17,689 [INFO] Starting task: Transform transactions
2026-07-11 23:58:17,689 [INFO] Starting transaction transformation
2026-07-11 23:58:17,698 [INFO] Transformation completed. Removed 0 invalid transactions.
2026-07-11 23:58:17,699 [INFO] Completed task: Transform transactions in 0.01 seconds
2026-07-11 23:58:17,699 [INFO] Starting task: Validate transactions
2026-07-11 23:58:17,699 [INFO] Starting transaction validation
2026-07-11 23:58:17,701 [INFO] PASS: transaction_id is unique
2026-07-11 23:58:17,701 [INFO] PASS: all amounts are positive
2026-07-11 23:58:17,702 [INFO] PASS: all timestamps are

,transaction_id,account_id,transaction_type,amount,channel,timestamp,is_flagged,amount_bucket,txn_hour
0,TXN0000001,ACC1014,deposit,388.59,mobile_app,2025-07-28 03:42:48.899833128,False,small,3
1,TXN0000002,ACC1030,payment,8387.23,atm,2025-03-18 16:32:20.435683925,False,medium,16
2,TXN0000003,ACC1063,withdrawal,1155.79,atm,2025-09-28 22:17:50.626068118,False,medium,22
3,TXN0000004,ACC1081,withdrawal,806.73,branch,2025-01-10 04:44:38.654277250,False,small,4
4,TXN0000005,ACC1076,deposit,740.60,mobile_app,2025-05-10 12:19:37.396409180,False,small,12


### Block D — Mini-Capstone: Multi-Source Pipeline (⭐⭐⭐ 60 min)
**Scenario:** Management wants a single `daily_summary` table combining all three sources.

Build a pipeline that:
1. Ingests the customers API (with fallback), the sales CSV, and the transactions Parquet
2. Produces one summary table: per city → number of customers, total sales revenue (via the toy customer_id join from earlier), and total transaction amount
3. Runs at least 4 data quality checks before loading
4. Loads the result into SQLite and exports `data/daily_summary.parquet`
5. Ends with a markdown cell: a 3-sentence "runbook" note describing what the pipeline does and how to rerun it.

In [35]:
# Block D — your capstone here
import pandas as pd
import sqlite3
import logging
import time
import json

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("daily_summary_pipeline")


def task(name, fn):
    """Run one pipeline task with logging and error handling."""

    start_time = time.time()
    log.info(f"Starting task: {name}")

    try:
        result = fn()
        elapsed = time.time() - start_time
        log.info(f"Completed task: {name} in {elapsed:.2f} seconds")
        return result

    except Exception:
        log.exception(f"Task failed: {name}")
        raise


def ingest_customers(api_url: str, fallback_path: str) -> pd.DataFrame:
    """Ingest customer data from an API with a local fallback."""

    try:
        import requests

        response = requests.get(api_url, timeout=10)
        response.raise_for_status()
        payload = response.json()

        records = payload["data"] if isinstance(payload, dict) and "data" in payload else payload
        customers = pd.json_normalize(records)

        required_columns = {"customer_id", "city"}

        if not required_columns.issubset(customers.columns):
            raise ValueError("API response does not contain customer_id and city")

        log.info(f"Loaded {len(customers)} customers from the live API")

    except Exception as error:
        log.warning(f"API ingestion failed. Using fallback file instead: {error}")

        with open(fallback_path) as file:
            payload = json.load(file)

        customers = pd.json_normalize(payload["data"])
        log.info(f"Loaded {len(customers)} customers from {fallback_path}")

    return customers


def ingest_sales(file_path: str) -> pd.DataFrame:
    """Ingest sales data from a CSV file."""

    sales = pd.read_csv(file_path)
    log.info(f"Loaded {len(sales)} sales records from {file_path}")

    return sales


def ingest_transactions(file_path: str) -> pd.DataFrame:
    """Ingest transaction data from a Parquet file."""

    transactions = pd.read_parquet(file_path)
    log.info(f"Loaded {len(transactions)} transaction records from {file_path}")

    return transactions


def transform_sources(customers: pd.DataFrame, sales: pd.DataFrame, transactions: pd.DataFrame) -> pd.DataFrame:
    """Clean, join, and summarize all three data sources."""

    customers = customers.copy()
    sales = sales.copy()
    transactions = transactions.copy()

    customers["customer_id"] = pd.to_numeric(customers["customer_id"], errors="coerce")
    customers["city"] = customers["city"].astype("string").str.strip()
    customers = customers.dropna(subset=["customer_id", "city"])
    customers = customers.drop_duplicates(subset=["customer_id"])
    customers["customer_id"] = customers["customer_id"].astype(int)

    sales = sales.drop_duplicates().reset_index(drop=True)
    sales["region"] = sales["region"].str.strip().str.upper()
    sales["order_date"] = pd.to_datetime(sales["order_date"], format="mixed", errors="coerce")

    missing_price = sales["unit_price"].isna()
    sales.loc[missing_price, "unit_price"] = (sales.loc[missing_price, "total_amount"] / sales.loc[missing_price, "quantity"]).round(2)

    customer_ids = customers["customer_id"].sort_values().tolist()

    if not customer_ids:
        raise ValueError("No valid customer IDs are available for the joins")

    # Toy sales join from the earlier notebook section
    sales["customer_id"] = [customer_ids[index % len(customer_ids)] for index in range(len(sales))]

    sales_enriched = sales.merge(customers[["customer_id", "city"]], on="customer_id", how="left")

    # Toy transaction join: ACC1000 becomes customer_id 1
    transactions["customer_id"] = pd.to_numeric(transactions["account_id"].str.extract(r"(\d+)$")[0], errors="coerce") - 999
    transactions["customer_id"] = transactions["customer_id"].astype("Int64")

    transactions_enriched = transactions.merge(customers[["customer_id", "city"]], on="customer_id", how="left")

    matched_transactions = transactions_enriched["city"].notna().sum()
    log.info(f"Matched {matched_transactions} of {len(transactions_enriched)} transactions to customers")

    customer_summary = customers.groupby("city", as_index=False).agg(customer_count=("customer_id", "nunique"))

    sales_summary = sales_enriched.groupby("city", as_index=False).agg(total_sales_revenue=("total_amount", "sum"))

    transaction_summary = (
        transactions_enriched
        .dropna(subset=["city"])
        .groupby("city", as_index=False)
        .agg(total_transaction_amount=("amount", "sum"))
    )

    daily_summary = customer_summary.merge(sales_summary, on="city", how="left")
    daily_summary = daily_summary.merge(transaction_summary, on="city", how="left")

    daily_summary["total_sales_revenue"] = daily_summary["total_sales_revenue"].fillna(0).round(2)
    daily_summary["total_transaction_amount"] = daily_summary["total_transaction_amount"].fillna(0).round(2)
    daily_summary = daily_summary.sort_values("city").reset_index(drop=True)

    log.info(f"Created daily summary containing {len(daily_summary)} cities")

    return daily_summary


def validate_daily_summary(df: pd.DataFrame) -> bool:
    """Run data quality checks before loading."""

    log.info("Starting daily summary validation")

    checks = {
        "summary is not empty": not df.empty,
        "city has no missing values": df["city"].notna().all(),
        "city values are not blank": df["city"].astype(str).str.strip().ne("").all(),
        "one row exists per city": df["city"].is_unique,
        "customer counts are positive": (df["customer_count"] > 0).all(),
        "sales revenue is non-negative": (df["total_sales_revenue"] >= 0).all(),
        "transaction amounts are non-negative": (df["total_transaction_amount"] >= 0).all(),
        "summary contains no missing values": not df.isna().any().any()
    }

    failed_checks = []

    for check_name, passed in checks.items():
        if passed:
            log.info(f"PASS: {check_name}")
        else:
            log.error(f"FAIL: {check_name}")
            failed_checks.append(check_name)

    if failed_checks:
        raise ValueError("Daily summary validation failed: " + ", ".join(failed_checks))

    log.info("All daily summary validation checks passed")

    return True


def load_daily_summary(df: pd.DataFrame, db_path: str, table_name: str, parquet_path: str) -> int:
    """Load the summary into SQLite and save a Parquet copy."""

    with sqlite3.connect(db_path) as conn:
        df.to_sql(table_name, conn, if_exists="replace", index=False)

    df.to_parquet(parquet_path, index=False)

    log.info(f"Loaded {len(df)} rows into {db_path}:{table_name}")
    log.info(f"Saved the Parquet copy to {parquet_path}")

    return len(df)


def daily_summary_pipeline() -> pd.DataFrame:
    """Run the complete multi-source pipeline."""

    api_url = "https://jsonplaceholder.typicode.com/users"
    fallback_path = "data/customers_api.json"
    sales_path = "data/sales_data.csv"
    transactions_path = "data/transactions.parquet"
    database_path = "dep_warehouse.db"
    table_name = "daily_summary"
    parquet_path = "data/daily_summary.parquet"

    log.info("Daily summary pipeline started")

    customers = task("Ingest customers API", lambda: ingest_customers(api_url, fallback_path))
    sales = task("Ingest sales CSV", lambda: ingest_sales(sales_path))
    transactions = task("Ingest transactions Parquet", lambda: ingest_transactions(transactions_path))

    daily_summary = task("Transform and combine sources", lambda: transform_sources(customers, sales, transactions))

    task("Validate daily summary", lambda: validate_daily_summary(daily_summary))

    task(
        "Load daily summary",
        lambda: load_daily_summary(daily_summary, database_path, table_name, parquet_path)
    )

    log.info("Daily summary pipeline completed successfully")

    return daily_summary


daily_summary = daily_summary_pipeline()

daily_summary

2026-07-13 11:48:42,058 [INFO] Daily summary pipeline started
2026-07-13 11:48:42,060 [INFO] Starting task: Ingest customers API
2026-07-13 11:48:42,420 [WARNING] API ingestion failed. Using fallback file instead: API response does not contain customer_id and city
2026-07-13 11:48:42,425 [INFO] Loaded 60 customers from data/customers_api.json
2026-07-13 11:48:42,427 [INFO] Completed task: Ingest customers API in 0.37 seconds
2026-07-13 11:48:42,427 [INFO] Starting task: Ingest sales CSV
2026-07-13 11:48:42,440 [INFO] Loaded 512 sales records from data/sales_data.csv
2026-07-13 11:48:42,441 [INFO] Completed task: Ingest sales CSV in 0.01 seconds
2026-07-13 11:48:42,441 [INFO] Starting task: Ingest transactions Parquet
2026-07-13 11:48:42,493 [INFO] Loaded 800 transaction records from data/transactions.parquet
2026-07-13 11:48:42,494 [INFO] Completed task: Ingest transactions Parquet in 0.05 seconds
2026-07-13 11:48:42,494 [INFO] Starting task: Transform and combine sources
2026-07-13 11

,city,customer_count,total_sales_revenue,total_transaction_amount
0,Cebu City,16,1031527.70,221230.75
1,Davao City,13,791845.43,258347.30
2,Makati,12,776940.38,232801.10
3,Quezon City,10,647949.37,137185.23
4,Taguig,9,568280.52,200924.66


This pipeline ingests customer, sales, andtransaction data, 
combines them using simulated customer IDs, and produces one summary row per city. 
Before loading, it checks that the result is complete, unique by city, 
and contains valid non-negative values. To rerun the pipeline, run the setup cells first
and then execute the Block D cell containing `daily_summary_pipeline()`.

---
---
### © Data Engineering Pilipinas (DEP). All rights reserved.
*This course material was created by and is the property of **Data Engineering Pilipinas**. It may not be copied, distributed, modified, or used — in whole or in part — without prior written consent from Data Engineering Pilipinas.*